# **BERT Text Similarity Search Engine**

##Loading the Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
folder_path = "/content/drive/MyDrive/datasets/md_files"
import os

texts = []
for file in os.listdir(folder_path):
    if file.endswith(".md"):
        with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
            texts.append(f.read())

print("Loaded", len(texts), "markdown files")


Loaded 15 markdown files


In [ ]:
import re

def clean_markdown(text):
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)
    text = re.sub(r'[#>*_`]', '', text)
    text = re.sub(r'\n+', ' ', text)
    return text.strip()

texts = [clean_markdown(t) for t in texts]


In [ ]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:

from nltk.tokenize import sent_tokenize

sentences = []
for text in texts:
    sentences.extend(sent_tokenize(text))

print("Total sentences:", len(sentences))


Total sentences: 7971


In [ ]:
import torch
from sentence_transformers.util import cos_sim


In [ ]:
!pip install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


## Implementing sentence embeddings

In [ ]:
sentence_embeddings = model.encode(
    sentences,         # your list of sentences
    convert_to_tensor=True,
    show_progress_bar=True
)


Batches:   0%|          | 0/250 [00:00<?, ?it/s]

In [ ]:
def semantic_search(query, top_k=5):
    query_embedding = model.encode(
        query,
        convert_to_tensor=True
    ).to(sentence_embeddings.device)

    similarity_scores = cos_sim(query_embedding, sentence_embeddings)[0]

    top_results = torch.topk(similarity_scores, k=top_k)

    results = []
    for score, idx in zip(top_results.values, top_results.indices):
        results.append({
            "sentence": sentences[idx],
            "score": float(score)
        })

    return results


## Finding top 5 similar sentence using cosine similarity

In [ ]:
while True:
    query = input("\nEnter your query (or type 'exit'): ")

    if query.lower() == "exit":
        print("Exiting search.")
        break

    results = semantic_search(query)

    print("\nTop 5 Similar Documents:\n")
    for i, res in enumerate(results, 1):
        print(f"{i}. Similarity Score: {res['score']:.4f}")
        print(f"   {res['sentence']}\n")



Enter your query (or type 'exit'): decision tree

Top 5 Similar Documents:

1. Similarity Score: 0.8238
   Decision Tree Classification Task !

2. Similarity Score: 0.7866
   "The decision tree classifier: Design and potential".

3. Similarity Score: 0.6848
   [](page1Figure3.jpeg) Figure 1: A simple decision tree (binary splits).

4. Similarity Score: 0.6742
   [](page245Figure1.jpeg)  Decision Tree Classification Task !

5. Similarity Score: 0.6677
   "Random decision forests".


Enter your query (or type 'exit'): cloud computing

Top 5 Similar Documents:

1. Similarity Score: 0.7347
   Understand and appreciate what is meant by the term Cloud Computing.

2. Similarity Score: 0.7088
   Before we start with the concepts of cloud, let us take a look at how cloud computing came about.

3. Similarity Score: 0.6868
   Cloud helps achieve the concept of utility computing in which computing services is offered as an utility in a similar manner as any other utility services such as electric